# ALCF Filesystem Operations via the AmSC Python Client

This notebook demonstrates how to interact with ALCF filesystems
using the AmSC Python Client's facility integration.

**What you'll do:**
1. Connect to ALCF and select a resource (Eagle storage or Polaris compute)
2. List directory contents
3. Read file content (head, tail, view)
4. Get file metadata (stat, checksum)
5. Perform file operations (mkdir, cp, mv, rm)

**Prerequisites:**
- `amsc-client` installed (with `globus-sdk`)
- A valid ALCF account
- A Globus account linked to your ALCF identity

**How it works:**
All filesystem operations are **asynchronous** — they run on the facility
via Globus Compute and return a `Task` object immediately. Call
`task.wait()` to block until completion, then access `task.result`
for the output.

```python
task = resource.fs.ls("/home/user")
task.wait()           # blocks until complete
result = task.result  # directory listing
```

**⚠️ Note:** The ALCF IRI Facility API is under active development.
Some filesystem endpoints may not be implemented yet — cells that
hit unimplemented endpoints will print a friendly message instead
of a stack trace.

In [ ]:
from amsc_client import Client, Task, ApiError, AuthenticationError, NotFoundError

## Step 1: Connect to ALCF

Create a client and connect to the ALCF facility. No auth is needed
to list resources — auth is triggered lazily on the first filesystem call.

In [ ]:
client = Client(token="not-needed-for-facilities")
alcf = client.facility("alcf")

# List resources — filesystem works on both compute and storage types
for r in alcf.resources():
    print(f"{r.name:12s}  type={r.resource_type:10s}  status={r.status}")

## Step 2: Select a Resource

Filesystem operations can target any resource:
- **Eagle** — ALCF's shared storage system (`type=storage`)
- **Polaris**, **Aurora**, etc. — compute resources that also have filesystem access

The `.fs` property gives you a `FilesystemClient` scoped to that resource.

In [ ]:
RESOURCE_NAME = "Eagle"

resource = alcf.resource(RESOURCE_NAME)
print(f"Resource: {resource.name} (type={resource.resource_type}, id={resource.id})")
print(f"Filesystem client: {resource.fs}")

## Step 3: List Directory Contents

The first filesystem call triggers a **Globus login** — a browser window
will open for ALCF authentication.

In [ ]:
ALCF_USERNAME = "parton"  # Change to your ALCF username
HOME_DIR = f"/home/{ALCF_USERNAME}"
JOB_DIR = f"{HOME_DIR}/iri_job_outputs"

try:
    task = resource.fs.ls(JOB_DIR)
    task.wait(timeout=60)

    files = task.result.get("output", []) if isinstance(task.result, dict) else []
    print(f"Found {len(files)} entries in {JOB_DIR}:")
    for f in files:
        print(f"  {f['permissions']}  {f['size']:>6s}  {f['last_modified'][:10]}  {f['name']}")
except ApiError as e:
    print(f"🚧 ls: Not available yet — {e}")

## Step 4: Read File Content

Three methods for reading files:

| Method | Description |
|--------|-------------|
| `head(path, lines=N)` | Read first N lines |
| `tail(path, lines=N)` | Read last N lines |
| `view(path, size=N, offset=M)` | Read N bytes starting at offset M |

In [ ]:
TEST_FILE = f"{JOB_DIR}/amsc-tutorial-20260330-143839.stdout"

try:
    task = resource.fs.head(TEST_FILE, lines=10)
    task.wait(timeout=60)
    print(f"head — first 10 lines:\n{task.result}")
except ApiError as e:
    print(f"🚧 head: Not available yet — {e}")

In [ ]:
try:
    task = resource.fs.tail(TEST_FILE, lines=5)
    task.wait(timeout=60)
    print(f"tail — last 5 lines:\n{task.result}")
except ApiError as e:
    print(f"🚧 tail: Not available yet — {e}")

In [ ]:
try:
    task = resource.fs.view(TEST_FILE, size=256, offset=0)
    task.wait(timeout=60)
    print(f"view — first 256 bytes:\n{task.result}")
except ApiError as e:
    print(f"🚧 view: Not available yet — {e}")

## Step 5: File Metadata

Get detailed file information with `stat()`, detect file types
with `file()`, and compute checksums with `checksum()`.

In [ ]:
try:
    task = resource.fs.stat(TEST_FILE)
    task.wait(timeout=60)
    print(f"stat:\n{task.result}")
except ApiError as e:
    print(f"🚧 stat: Not available yet — {e}")

In [ ]:
try:
    task = resource.fs.file(TEST_FILE)
    task.wait(timeout=60)
    print(f"file type: {task.result}")
except ApiError as e:
    print(f"🚧 file: Not available yet — {e}")

In [ ]:
try:
    task = resource.fs.checksum(TEST_FILE)
    task.wait(timeout=60)
    print(f"checksum: {task.result}")
except ApiError as e:
    print(f"🚧 checksum: Not available yet — {e}")

## Step 6: File Operations

Create directories, copy/move/delete files — all return `Task` objects.

**⚠️ Be careful with rm** — there's no undo!

In [ ]:
import datetime
RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
TEST_DIR = f"{HOME_DIR}/fs_tutorial_{RUN_ID}"

try:
    task = resource.fs.mkdir(TEST_DIR)
    task.wait(timeout=60)
    print(f"Created {TEST_DIR}: {task.state}")

    task = resource.fs.ls(TEST_DIR)
    task.wait(timeout=60)
    print(f"Directory contents: {task.result}")
except ApiError as e:
    print(f"🚧 mkdir: Not available yet — {e}")

In [ ]:
dest = f"{TEST_DIR}/copied_file.txt"

try:
    task = resource.fs.cp(TEST_FILE, dest)
    task.wait(timeout=60)
    print(f"Copied → {dest}: {task.state}")

    task = resource.fs.ls(TEST_DIR)
    task.wait(timeout=60)
    files = task.result.get("output", []) if isinstance(task.result, dict) else []
    for f in files:
        print(f"  {f['name']}  ({f['size']} bytes)")
except ApiError as e:
    print(f"🚧 cp: Not available yet — {e}")

In [ ]:
old_path = f"{TEST_DIR}/copied_file.txt"
new_path = f"{TEST_DIR}/renamed_file.txt"

try:
    task = resource.fs.mv(old_path, new_path)
    task.wait(timeout=60)
    print(f"Moved → {new_path}: {task.state}")

    task = resource.fs.ls(TEST_DIR)
    task.wait(timeout=60)
    files = task.result.get("output", []) if isinstance(task.result, dict) else []
    for f in files:
        print(f"  {f['name']}")
except ApiError as e:
    print(f"🚧 mv: Not available yet — {e}")

In [ ]:
try:
    task = resource.fs.rm(TEST_DIR)
    task.wait(timeout=60)
    print(f"Removed {TEST_DIR}: {task.state}")
except ApiError as e:
    print(f"🚧 rm: Not available yet — {e}")

## Step 7: Permission & Archive Operations

In [ ]:
try:
    task = resource.fs.chmod(TEST_FILE, "644")
    task.wait(timeout=60)
    print(f"chmod 644: {task.state}")
except ApiError as e:
    print(f"🚧 chmod: Not available yet — {e}")

In [ ]:
LINK_PATH = f"{HOME_DIR}/test_link_{RUN_ID}"

try:
    task = resource.fs.symlink(TEST_FILE, LINK_PATH)
    task.wait(timeout=60)
    print(f"symlink → {LINK_PATH}: {task.state}")

    # Clean up
    task = resource.fs.rm(LINK_PATH)
    task.wait(timeout=60)
except ApiError as e:
    print(f"🚧 symlink: Not available yet — {e}")

In [ ]:
ARCHIVE = f"{HOME_DIR}/test_archive_{RUN_ID}.tar.gz"
EXTRACT_DIR = f"{HOME_DIR}/test_extract_{RUN_ID}"

try:
    task = resource.fs.compress(JOB_DIR, ARCHIVE)
    task.wait(timeout=120)
    print(f"compress: {task.state}")

    task = resource.fs.extract(ARCHIVE, EXTRACT_DIR)
    task.wait(timeout=120)
    print(f"extract: {task.state}")

    # Clean up
    resource.fs.rm(ARCHIVE).wait(timeout=60)
    resource.fs.rm(EXTRACT_DIR).wait(timeout=60)
except ApiError as e:
    print(f"🚧 compress/extract: Not available yet — {e}")

## Step 8: Task Management

Every filesystem operation returns a `Task` object. You can inspect
and manage tasks independently.

In [ ]:
import time

task = resource.fs.ls(HOME_DIR)

print(f"Task ID:    {task.id}")
print(f"Task URI:   {task.uri}")
print(f"State:      {task.state}")
print(f"Terminal:   {task.is_terminal}")

# Manual polling loop
for i in range(15):
    state = task.status  # calls the API
    print(f"  Poll {i+1}: {state}")
    if task.is_terminal:
        break
    time.sleep(2)

print(f"\nFinal state: {task.state}")
print(f"Command:     {task.command}")
if task.result:
    entries = task.result.get("output", []) if isinstance(task.result, dict) else []
    print(f"Result:      {len(entries)} entries")

## Summary

| Operation | API Call | Returns |
|-----------|---------|--------|
| List directory | `resource.fs.ls(path)` | Task → directory listing |
| File status | `resource.fs.stat(path)` | Task → file metadata |
| Read head | `resource.fs.head(path, lines=N)` | Task → file content |
| Read tail | `resource.fs.tail(path, lines=N)` | Task → file content |
| Read bytes | `resource.fs.view(path, size=N)` | Task → file content |
| File type | `resource.fs.file(path)` | Task → type string |
| Checksum | `resource.fs.checksum(path)` | Task → checksum string |
| Create dir | `resource.fs.mkdir(path)` | Task |
| Remove | `resource.fs.rm(path)` | Task |
| Copy | `resource.fs.cp(src, dst)` | Task |
| Move | `resource.fs.mv(src, dst)` | Task |
| Symlink | `resource.fs.symlink(target, link)` | Task |
| Permissions | `resource.fs.chmod(path, '755')` | Task |
| Ownership | `resource.fs.chown(path, owner='user')` | Task |
| Compress | `resource.fs.compress(src, dst)` | Task |
| Extract | `resource.fs.extract(src, dst)` | Task |

**Key concepts:**
- All operations are **asynchronous** via Globus Compute
- Every call returns a **Task** — use `task.wait()` then `task.result`
- **Resource-scoped**: `resource.fs.ls(...)` — operations target a specific resource
- Works on both **compute** (Polaris) and **storage** (Eagle) resources
- Auth is **lazy** — Globus login only triggers on first filesystem call
- 🚧 marks endpoints the ALCF API hasn't implemented yet